# Benchmark MinHashLSH on arXiv Multi-Parser Corpus

Đánh giá **Precision / Recall / F1** của thuật toán **MinHash LSH** trên bộ benchmark near-duplicate được xây dựng từ dữ liệu arXiv tự crawl với 4 phương pháp parse khác nhau.


| Bước | Mô tả |
|---|---|
| **1. Sample originals** (`data_list`) | Mỗi `doc_id` được gán ngẫu nhiên cho **một** parser (random choice trong các parsers có sẵn), không yêu cầu có mặt ở tất cả 4 parsers. `modification=0` |
| **2. Parser duplicates** (`data_dupl_list`) | Với một tập con `doc_id`, tìm bản của **parser khác** → `modification=1`. Schema: `text` = bản parser kia, `new_text` = `text` |
| **3. Truncation duplicates** (`data_trunc_list`) | Lấy original, cắt 1–20% nội dung tại vị trí ngẫu nhiên → `modification=2`. Schema: `text` = gốc, `new_text` = đã cắt |
| **4. n_trunc formula** | Repo: `n_trunc = ((n_dupl*(p-1)) + (n_orig*p)) / (1-p)` — tính **sau** khi biết số parser dup thực tế |
| **5. Label** | Sau shuffle: lần xuất hiện đầu của `doc_id` → `is_duplicate=0`, lần sau → `is_duplicate=1` |
| **6. Dedup dùng `new_text`** | `run_minhash_lsh_stream` hash theo `new_text` (truncated cho mod=2, original cho mod=0/1) |

## Nguồn dữ liệu
- `arxiv_html` — HTML full-text từ arXiv (reference)
- `pymupdf` — PDF text-layer (PyMuPDF)
- `pypdf` — PDF text-layer (pypdf)
- `tesseract_ocr` — OCR bằng Tesseract 5.x

## Tuning / Test split
- **20% doc_ids** cho tuning (3 prevalences: 0.3, 0.5, 0.7)
- **80% doc_ids** cho test (6 prevalences: 0.1, 0.2, 0.3, 0.5, 0.7, 0.9)

In [1]:
import os

# ── Paths to parsed JSONL files ───────────────────────────────────────────────
_ABS = r"d:\Project\LSHBloomExperiments\notebook"
HTML_JSONL      = os.path.join(_ABS, "arxiv_html_reference_workspace_v2", "parsed_jsonl", "arxiv_html_reference_v2_neardup.jsonl")
PYMUPDF_JSONL   = os.path.join(_ABS, "arxiv_ocr_benchmark_workspace",   "parsed_jsonl", "pymupdf.jsonl")
PYPDF_JSONL     = os.path.join(_ABS, "arxiv_ocr_benchmark_workspace",   "parsed_jsonl", "pypdf.jsonl")
TESSERACT_JSONL = os.path.join(_ABS, "arxiv_tesseract_ocr_benchmark_workspace", "parsed_jsonl", "arxiv_tesseract_ocr_all.jsonl")

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(_ABS, "..", "minhashlsh_results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Data filters ──────────────────────────────────────────────────────────────
MIN_CHARS = 500          # minimum text length (chars) to include a document

# ── Train/test split ──────────────────────────────────────────────────────────
TUNING_FRAC = 0.20
RANDOM_SEED = 42

# ── Hyperparameter search grid ────────────────────────────────────────────────
THRESHOLD_GRID = [0.5, 0.6, 0.7, 0.8, 0.9]
NUM_PERM_GRID  = [64, 128, 256]
NGRAM_GRID     = [1, 3]

# ── Prevalence grids ─────────────────────────────────────────────────────────
TUNING_PREVALENCE_GRID = [0.3, 0.5, 0.7]
PREVALENCE_GRID        = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]

# ── Truncation params ─────────────────────────────────────────────────────────
VALID_PERCENTAGES    = [1, 2, 5, 7, 10, 15, 20]
VALID_REL_LOCATIONS  = [0, 0.25, 0.5, 0.75, 1.0]

# ── Parser priority ───────────────────────────────────────────────────────────
PARSER_PRIORITY = ["arxiv_html", "pymupdf", "pypdf", "tesseract_ocr"]

print("Configuration loaded.")
print(f"HTML JSONL  : {HTML_JSONL}")
print(f"  exists    : {os.path.exists(HTML_JSONL)}")
print(f"Output dir  : {os.path.abspath(OUTPUT_DIR)}")
print(f"Tuning frac : {TUNING_FRAC:.0%}  |  Test frac: {1-TUNING_FRAC:.0%}")


Configuration loaded.
HTML JSONL  : d:\Project\LSHBloomExperiments\notebook\arxiv_html_reference_workspace_v2\parsed_jsonl\arxiv_html_reference_v2_neardup.jsonl
  exists    : True
Output dir  : d:\Project\LSHBloomExperiments\minhashlsh_results
Tuning frac : 20%  |  Test frac: 80%


In [2]:
import json
import random
import time
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from datasketch import MinHash, MinHashLSH
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    classification_report, roc_auc_score, balanced_accuracy_score,
)

warnings.filterwarnings("ignore")
print("All imports OK.")


All imports OK.


In [3]:
PARSER_FILES = {
    "arxiv_html":    HTML_JSONL,
    "pymupdf":       PYMUPDF_JSONL,
    "pypdf":         PYPDF_JSONL,
    "tesseract_ocr": TESSERACT_JSONL,
}

corpus = {}   # { parser_name: { doc_id: text } }

for parser_name, path in PARSER_FILES.items():
    corpus[parser_name] = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            doc_id = d["doc_id"]
            text   = d.get("text") or ""
            if d.get("fetch_status", "ok") != "ok":
                continue
            if len(text) >= MIN_CHARS:
                corpus[parser_name][doc_id] = text
    print(f"  {parser_name:20s}: {len(corpus[parser_name]):,} valid docs loaded")

# ── Valid doc_ids: UNION across all parsers (repo style — no hard intersection) ──
# A doc_id qualifies if it appears in AT LEAST ONE parser with valid text.
# This maximises dataset size; cross-parser dups will only be generated when
# the doc_id is actually present in multiple parsers.
all_id_sets  = [set(corpus[p].keys()) for p in PARSER_PRIORITY]
union_ids    = set.union(*all_id_sets)
common_ids   = set.intersection(*all_id_sets)
VALID_DOC_IDS = sorted(union_ids)

print(f"\nDoc_ids in ANY parser   : {len(VALID_DOC_IDS):,}  ← used for benchmark (repo style)")
print(f"Doc_ids in ALL 4 parsers: {len(common_ids):,}  (old common_ids approach for reference)")

# Per-pair overlap (useful to know how rich cross-parser dups will be)
pairs = [("arxiv_html","pymupdf"), ("arxiv_html","tesseract_ocr"),
         ("pymupdf","pypdf"), ("pymupdf","tesseract_ocr")]
print("\nCross-parser doc_id overlap (determines parser-dup candidates):")
for pa, pb in pairs:
    n = len(set(corpus[pa]) & set(corpus[pb]))
    print(f"  {pa} ∩ {pb:20s}: {n:,}")


  arxiv_html          : 1,413 valid docs loaded
  pymupdf             : 1,578 valid docs loaded
  pypdf               : 1,578 valid docs loaded
  tesseract_ocr       : 1,578 valid docs loaded

Doc_ids in ANY parser   : 1,578  ← used for benchmark (repo style)
Doc_ids in ALL 4 parsers: 1,413  (old common_ids approach for reference)

Cross-parser doc_id overlap (determines parser-dup candidates):
  arxiv_html ∩ pymupdf             : 1,413
  arxiv_html ∩ tesseract_ocr       : 1,413
  pymupdf ∩ pypdf               : 1,578
  pymupdf ∩ tesseract_ocr       : 1,578


In [4]:
# ── Per-parser text-length statistics ────────────────────────────────────────
rows = []
for pname in PARSER_PRIORITY:
    lengths = [len(t) for t in corpus[pname].values()]
    rows.append({
        "parser":        pname,
        "n_docs":        len(corpus[pname]),
        "mean_chars":    int(np.mean(lengths)),
        "median_chars":  int(np.median(lengths)),
        "min_chars":     int(np.min(lengths)),
        "max_chars":     int(np.max(lengths)),
    })
stats_df = pd.DataFrame(rows)
print("=== Parser Coverage & Text-Length Stats ===")
display(stats_df.to_string(index=False))

# ── Cross-parser Jaccard similarity (word-unigram, only for doc_ids in BOTH parsers) ──
def word_jaccard(ta: str, tb: str) -> float:
    sa, sb = set(ta.lower().split()), set(tb.lower().split())
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

rng_sample = random.Random(42)
# sample from ALL valid ids; filter per-pair to those actually present in both parsers
sample_ids_all = rng_sample.sample(VALID_DOC_IDS, min(500, len(VALID_DOC_IDS)))

print("\n=== Cross-parser Word-Jaccard Similarity ===")
pairs = [("arxiv_html","pymupdf"), ("arxiv_html","tesseract_ocr"),
         ("pymupdf","pypdf"),      ("pymupdf","tesseract_ocr")]
for pa, pb in pairs:
    # only compute where both parsers have the doc
    shared = [d for d in sample_ids_all if d in corpus[pa] and d in corpus[pb]]
    if not shared:
        print(f"  {pa} vs {pb}: no overlap in sample")
        continue
    jacs = [word_jaccard(corpus[pa][d], corpus[pb][d]) for d in shared]
    print(f"  {pa} vs {pb:20s} (n={len(shared):3d}): "
          f"mean={np.mean(jacs):.3f}  median={np.median(jacs):.3f}  "
          f"p10={np.percentile(jacs,10):.3f}  p90={np.percentile(jacs,90):.3f}")


=== Parser Coverage & Text-Length Stats ===


'       parser  n_docs  mean_chars  median_chars  min_chars  max_chars\n   arxiv_html    1413       60648         53381       8141     671448\n      pymupdf    1578       66557         56939       8487     928306\n        pypdf    1578       66648         56807       8387     924182\ntesseract_ocr    1578       62960         54566       7506     903859'


=== Cross-parser Word-Jaccard Similarity ===
  arxiv_html vs pymupdf              (n=449): mean=0.630  median=0.632  p10=0.501  p90=0.759
  arxiv_html vs tesseract_ocr        (n=449): mean=0.531  median=0.535  p10=0.400  p90=0.646
  pymupdf vs pypdf                (n=500): mean=0.844  median=0.858  p10=0.730  p90=0.940
  pymupdf vs tesseract_ocr        (n=500): mean=0.644  median=0.660  p10=0.476  p90=0.788


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# Core utility functions
# ═══════════════════════════════════════════════════════════════════════════════

def truncate(s: str, percentage: float, rel_location: float) -> str:
    """
    Truncate `percentage`% of characters from string s at `rel_location`.
    Exact replica of the author's truncate() in dedup_benchmark_utils.py.

    percentage    : one of [1, 2, 5, 7, 10, 15, 20]
    rel_location  : one of [0, 0.25, 0.5, 0.75, 1.0]
    """
    total_chars   = len(s)
    chars_to_remove = int((percentage / 100) * total_chars)
    start_pos     = int(total_chars * rel_location)
    start_truncate = max(0, start_pos - chars_to_remove // 2)
    end_truncate   = min(total_chars, start_truncate + chars_to_remove)
    return s[:start_truncate] + s[end_truncate:]


def make_shingles(text: str, ngram_n: int = 1) -> list:
    """
    Tokenise text into word-level shingles.

    ngram_n=1  →  unique word unigrams  (author's default in lsh.py)
    ngram_n>1  →  contiguous word n-grams (deduped)
    """
    words = text.lower().split()
    if not words:
        return [""]
    if ngram_n <= 1:
        return list(set(words))          # unique unigrams
    if len(words) < ngram_n:
        return [" ".join(words)]
    seen, out = set(), []
    for i in range(len(words) - ngram_n + 1):
        sh = " ".join(words[i : i + ngram_n])
        if sh not in seen:
            out.append(sh)
            seen.add(sh)
    return out


def compute_minhash(text: str, num_perm: int, ngram_n: int) -> MinHash:
    mh = MinHash(num_perm=num_perm)
    for sh in make_shingles(text, ngram_n):
        mh.update(sh.encode("utf-8"))
    return mh


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return {
        "precision":    precision_score(y_true, y_pred, zero_division=0),
        "recall":       recall_score(y_true, y_pred, zero_division=0),
        "f1":           f1_score(y_true, y_pred, zero_division=0),
        "tp": tp, "fp": fp, "fn": fn,
    }


print("Core functions defined  ✓")


Core functions defined  ✓


In [10]:
def build_benchmark_stream(
    corpus: dict,
    doc_ids: list,
    prevalence: float,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Parser-centric benchmark builder mirroring gen_dedup_benchmark.py + dedup_benchmark_utils.py.

    Steps
    ─────
    1. Parser-centric originals (mod=0):
       Each doc_id assigned to exactly ONE parser via balanced argmin assignment
       (parser with fewest originals so far that has that doc_id).
       Builds per_parser_originals dict — each parser owns its dedicated pool.

    2. Parser-centric parser-dups (mod=1):
       Mirrors collect_duplicates(): for each src_parser, take its candidate pool
       (up to per_parser_limit doc_ids), look up the same doc_id in each other parser.
       Apply global limit = num_planned_parser_dups.

    3. n_trunc — exact repo formula AFTER knowing actual parser-dup count:
         n_trunc = ((n_dupl*(p-1)) + (n_orig*p)) / (1-p)

    4. Truncation with guaranteed backfill (while loop — never leaves n_trunc under-filled).

    5. Merge + shuffle + label: first appearance of doc_id = 0, subsequent = 1.
    """
    rng     = random.Random(seed)
    parsers = [p for p in PARSER_PRIORITY if p in corpus]
    n_p     = len(parsers)

    doc_id_pool = list(doc_ids)
    rng.shuffle(doc_id_pool)

    # ── Step 1: Parser-centric originals ──────────────────────────────────────
    # Assign each doc_id to the parser with fewest originals so far (balanced).
    # Mirrors sample_parser_data(): claimed doc_ids are never double-assigned.
    per_parser_originals: dict = {p: [] for p in parsers}
    parser_counts: dict        = {p: 0  for p in parsers}
    claimed_ids: set           = set()

    for doc_id in doc_id_pool:
        avail = [
            p for p in parsers
            if doc_id in corpus.get(p, {}) and len(corpus[p][doc_id]) >= MIN_CHARS
        ]
        if not avail:
            continue
        # Balanced: assign to parser with fewest originals among available
        chosen = min(avail, key=lambda p: parser_counts[p])
        text   = corpus[chosen][doc_id]
        per_parser_originals[chosen].append({
            "doc_id":           doc_id,
            "text":             text,
            "new_text":         text,
            "parser":           chosen,
            "modification":     0,
            "trunc_percentage": 0,
            "rel_location":     0.0,
        })
        claimed_ids.add(doc_id)
        parser_counts[chosen] += 1

    data_list = [d for p in parsers for d in per_parser_originals[p]]
    n_orig    = len(data_list)

    # ── Plan limits ────────────────────────────────────────────────────────────
    num_planned_dups        = int(n_orig * ((1.0 / max(1e-9, 1.0 - prevalence)) - 1))
    num_planned_parser_dups = num_planned_dups // 2
    per_parser_limit        = max(1, num_planned_parser_dups // n_p)

    # ── Step 2: Parser-centric parser-dups ────────────────────────────────────
    # For each src_parser: take its candidate pool (up to per_parser_limit doc_ids).
    # For each other_parser: find same doc_id → add as mod=1 dup.
    # Mirrors collect_duplicates() pair-loop logic.
    data_dupl_list: list = []
    used_dup_ids:   set  = set()

    for src_parser in parsers:
        pool_ids = [d["doc_id"] for d in per_parser_originals[src_parser]]
        rng.shuffle(pool_ids)
        pool_ids = pool_ids[:per_parser_limit]

        for other_parser in parsers:
            if other_parser == src_parser:
                continue
            for doc_id in pool_ids:
                if doc_id in used_dup_ids:
                    continue
                if doc_id not in corpus.get(other_parser, {}):
                    continue
                text = corpus[other_parser][doc_id]
                if len(text) < MIN_CHARS:
                    continue
                data_dupl_list.append({
                    "doc_id":           doc_id,
                    "text":             text,
                    "new_text":         text,
                    "parser":           other_parser,
                    "modification":     1,
                    "trunc_percentage": 0,
                    "rel_location":     0.0,
                })
                used_dup_ids.add(doc_id)

    # Apply global limit and shuffle (mirrors repo's random.sample at the end)
    rng.shuffle(data_dupl_list)
    data_dupl_list = data_dupl_list[:num_planned_parser_dups]
    parser_dup_ids = {d["doc_id"] for d in data_dupl_list}

    # ── Step 3: n_trunc — exact repo formula ──────────────────────────────────
    # p = (n_dupl + n_trunc) / (n_orig + n_dupl + n_trunc)
    # → n_trunc = ((n_dupl*(p-1)) + (n_orig*p)) / (1-p)
    n_actual_dupl = len(data_dupl_list)
    n_trunc = int(
        ((n_actual_dupl * (prevalence - 1)) + (n_orig * prevalence))
        / max(1e-9, 1.0 - prevalence)
    )
    n_trunc = max(0, n_trunc)

    # ── Step 4: Truncation with guaranteed backfill ────────────────────────────
    # Prefer doc_ids not already used as parser dups (repo's available_paths logic).
    avail_trunc = [d for d in data_list if d["doc_id"] not in parser_dup_ids]
    if len(avail_trunc) < 50:
        avail_trunc = data_list.copy()   # repo fail-safe
    rng.shuffle(avail_trunc)

    # Extend pool until it has at least n_trunc slots (with-replacement style)
    pool  = avail_trunc.copy()
    while len(pool) < n_trunc:
        pool.extend(avail_trunc)

    data_trunc_list: list = []
    p_idx = 0
    while len(data_trunc_list) < n_trunc:
        if p_idx >= len(pool):
            pool.extend(avail_trunc)   # emergency extension — should rarely trigger
        orig     = pool[p_idx]
        p_idx   += 1
        pct      = rng.choice(VALID_PERCENTAGES)
        loc      = rng.choice(VALID_REL_LOCATIONS)
        new_text = truncate(orig["text"], pct, loc)
        if len(new_text) < MIN_CHARS // 4:
            continue   # skip degenerate truncations; pool extended above if needed
        data_trunc_list.append({
            "doc_id":           orig["doc_id"],
            "text":             orig["text"],   # original preserved (repo schema)
            "new_text":         new_text,        # truncated submitted doc
            "parser":           orig["parser"],
            "modification":     2,
            "trunc_percentage": pct,
            "rel_location":     loc,
        })

    # ── Step 5: Merge + shuffle ────────────────────────────────────────────────
    all_docs = data_list + data_dupl_list + data_trunc_list
    rng.shuffle(all_docs)

    # ── Step 6: Label — first appearance of doc_id = 0, repeated = 1 ──────────
    seen: set = set()
    for doc in all_docs:
        if doc["doc_id"] in seen:
            doc["is_duplicate"] = 1
        else:
            doc["is_duplicate"] = 0
            seen.add(doc["doc_id"])

    df = pd.DataFrame(all_docs)
    df["prevalence_target"] = prevalence
    df["stream_pos"]        = np.arange(len(df))
    actual_prev             = (df["is_duplicate"] == 1).sum() / max(1, len(df))
    df["prevalence_actual"] = round(actual_prev, 4)
    return df


print("build_benchmark_stream() defined  ✓  (parser-centric | balanced originals | guaranteed backfill)")


build_benchmark_stream() defined  ✓  (parser-centric | balanced originals | guaranteed backfill)


In [11]:
# ── Split doc_ids into tuning / test ─────────────────────────────────────────
rng_split = random.Random(RANDOM_SEED)
all_ids   = VALID_DOC_IDS.copy()
rng_split.shuffle(all_ids)

split_idx  = int(len(all_ids) * TUNING_FRAC)
tuning_ids = all_ids[:split_idx]
test_ids   = all_ids[split_idx:]

print(f"Total doc_ids : {len(all_ids):,}")
print(f"Tuning split  : {len(tuning_ids):,}  ({100*TUNING_FRAC:.0f}%)")
print(f"Test split    : {len(test_ids):,}  ({100*(1-TUNING_FRAC):.0f}%)")

# ── Build tuning streams ───────────────────────────────────────────────────────
print(f"\nBuilding tuning streams for prevalences: {TUNING_PREVALENCE_GRID} ...")
tuning_streams = {}
for i, prev in enumerate(TUNING_PREVALENCE_GRID):
    tuning_streams[prev] = build_benchmark_stream(corpus, tuning_ids, prev, seed=1000 + i)

# ── Build test streams ────────────────────────────────────────────────────────
print(f"Building test streams for prevalences: {PREVALENCE_GRID} ...")
test_streams = {}
for i, prev in enumerate(PREVALENCE_GRID):
    test_streams[prev] = build_benchmark_stream(corpus, test_ids, prev, seed=2000 + i)

# ── Sanity check ──────────────────────────────────────────────────────────────
def _stream_sanity(label: str, df: pd.DataFrame, target_prev: float):
    n_tot    = len(df)
    n_dup    = (df["is_duplicate"] == 1).sum()
    actual   = df["prevalence_actual"].iloc[0]
    mc       = df["modification"].value_counts().sort_index().to_dict()

    print(f"\n[{label}] prev_target={target_prev:.1f} | actual={actual:.3f} | "
          f"n={n_tot:,}  dup={n_dup:,}")
    print(f"  mod breakdown  → orig(0)={mc.get(0,0):,}  "
          f"parser_dup(1)={mc.get(1,0):,}  trunc_dup(2)={mc.get(2,0):,}")

    # Per-parser originals
    orig_by_parser = (
        df[df["modification"] == 0]
        .groupby("parser").size().reindex(PARSER_PRIORITY, fill_value=0)
    )
    print(f"  originals/parser → " +
          "  ".join(f"{p}:{orig_by_parser[p]}" for p in PARSER_PRIORITY if p in orig_by_parser))

    # Parser-dup target parser distribution
    dup_by_parser = (
        df[df["modification"] == 1]
        .groupby("parser").size().reindex(PARSER_PRIORITY, fill_value=0)
    )
    print(f"  parser_dups dst → " +
          "  ".join(f"{p}:{dup_by_parser[p]}" for p in PARSER_PRIORITY if p in dup_by_parser))


print("\n=== Stream sanity check ===")
for prev in TUNING_PREVALENCE_GRID:
    _stream_sanity("tuning", tuning_streams[prev], prev)

for prev in [0.3, 0.5, 0.7]:   # spot-check test
    _stream_sanity("test  ", test_streams[prev], prev)


Total doc_ids : 1,578
Tuning split  : 315  (20%)
Test split    : 1,263  (80%)

Building tuning streams for prevalences: [0.3, 0.5, 0.7] ...
Building test streams for prevalences: [0.1, 0.2, 0.3, 0.5, 0.7, 0.9] ...

=== Stream sanity check ===

[tuning] prev_target=0.3 | actual=0.300 | n=450  dup=135
  mod breakdown  → orig(0)=315  parser_dup(1)=64  trunc_dup(2)=71
  originals/parser → arxiv_html:79  pymupdf:79  pypdf:79  tesseract_ocr:78
  parser_dups dst → arxiv_html:41  pymupdf:20  pypdf:3  tesseract_ocr:0

[tuning] prev_target=0.5 | actual=0.500 | n=630  dup=315
  mod breakdown  → orig(0)=315  parser_dup(1)=156  trunc_dup(2)=159
  originals/parser → arxiv_html:79  pymupdf:79  pypdf:79  tesseract_ocr:78
  parser_dups dst → arxiv_html:101  pymupdf:51  pypdf:4  tesseract_ocr:0

[tuning] prev_target=0.7 | actual=0.700 | n=1,049  dup=734
  mod breakdown  → orig(0)=315  parser_dup(1)=315  trunc_dup(2)=419
  originals/parser → arxiv_html:79  pymupdf:79  pypdf:79  tesseract_ocr:78
  parser_

In [8]:
def run_minhash_lsh_stream(
    stream_df: pd.DataFrame,
    threshold: float,
    num_perm:  int,
    ngram_n:   int,
) -> tuple:
    """
    Stream evaluation of MinHashLSH.

    Mirrors the author's LSHDeduper.deduplicate() logic exactly:
      • Hash `new_text` (= truncated text for mod=2, original for mod=0/1)
        This matches the repo convention where new_text is the "submitted" document.
      • Query LSH → if any hit → predict duplicate (is_dup=1)
      • INSERT into LSH only when predicted as unique (first-seen cluster head)

    Returns
    -------
    metrics  : dict with precision, recall, f1, tp, fp, fn, timing
    y_true   : np.ndarray
    y_pred   : np.ndarray
    """
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

    y_true_list, y_pred_list = [], []
    t_query = t_insert = 0.0

    for i, row in enumerate(stream_df.itertuples(index=False)):
        # Use new_text: for mod=2 this is the truncated doc; for mod=0/1 it equals text.
        # This is exactly how the repo's JSONL for dedup is constructed.
        text = row.new_text
        key  = f"{row.doc_id}__{row.parser}__{i}"   # unique key per stream position

        mh = compute_minhash(text, num_perm, ngram_n)

        # ── Query ──────────────────────────────────────────────────────────
        t0    = time.perf_counter()
        hits  = lsh.query(mh)
        t_query += time.perf_counter() - t0

        is_dup_pred = 1 if hits else 0
        y_true_list.append(row.is_duplicate)
        y_pred_list.append(is_dup_pred)

        # ── Insert if predicted unique (author's deduplicate() logic) ──────
        if not is_dup_pred:
            t0 = time.perf_counter()
            lsh.insert(key, mh)
            t_insert += time.perf_counter() - t0

    y_true = np.array(y_true_list, dtype=np.int32)
    y_pred = np.array(y_pred_list, dtype=np.int32)

    metrics = compute_metrics(y_true, y_pred)
    metrics.update({
        "n_docs":            len(stream_df),
        "n_true_duplicates": int(y_true.sum()),
        "n_predicted_dups":  int(y_pred.sum()),
        "query_sec":         round(t_query,  4),
        "insert_sec":        round(t_insert, 4),
    })
    return metrics, y_true, y_pred


print("run_minhash_lsh_stream() defined  ✓  (hashes new_text)")


run_minhash_lsh_stream() defined  ✓  (hashes new_text)


In [9]:
print("=== Hyperparameter Tuning (Tuning Set — 3 prevalences) ===\n")

tuning_rows   = []
total_configs = len(NGRAM_GRID) * len(NUM_PERM_GRID) * len(THRESHOLD_GRID)
done          = 0

for ngram_n in NGRAM_GRID:
    for num_perm in NUM_PERM_GRID:
        for threshold in THRESHOLD_GRID:
            prev_f1s = []
            for prev, stream_df in tuning_streams.items():   # tuning_streams has TUNING_PREVALENCE_GRID
                metrics, _, _ = run_minhash_lsh_stream(stream_df, threshold, num_perm, ngram_n)
                tuning_rows.append({
                    "ngram_n":           ngram_n,
                    "num_perm":          num_perm,
                    "threshold":         threshold,
                    "prevalence":        prev,
                    "precision":         metrics["precision"],
                    "recall":            metrics["recall"],
                    "f1":                metrics["f1"],
                    "tp":                metrics["tp"],
                    "fp":                metrics["fp"],
                    "fn":                metrics["fn"],
                    "n_docs":            metrics["n_docs"],
                    "n_true_duplicates": metrics["n_true_duplicates"],
                    "query_sec":         metrics["query_sec"],
                    "insert_sec":        metrics["insert_sec"],
                })
                prev_f1s.append(metrics["f1"])

            done += 1
            mean_f1 = float(np.mean(prev_f1s))
            print(f"  [{done:2d}/{total_configs}] "
                  f"ng={ngram_n} perm={num_perm:3d} thr={threshold:.1f} "
                  f"→ mean_F1={mean_f1:.4f}")

tuning_df = pd.DataFrame(tuning_rows)
print(f"\nTuning complete — {len(tuning_rows)} rows recorded.")
print(f"({total_configs} configs × {len(TUNING_PREVALENCE_GRID)} tuning prevalences = {len(tuning_rows)} rows)")


=== Hyperparameter Tuning (Tuning Set — 3 prevalences) ===



  [ 1/30] ng=1 perm= 64 thr=0.5 → mean_F1=0.9150
  [ 2/30] ng=1 perm= 64 thr=0.6 → mean_F1=0.8541
  [ 3/30] ng=1 perm= 64 thr=0.7 → mean_F1=0.7847
  [ 4/30] ng=1 perm= 64 thr=0.8 → mean_F1=0.7106
  [ 5/30] ng=1 perm= 64 thr=0.9 → mean_F1=0.5142
  [ 6/30] ng=1 perm=128 thr=0.5 → mean_F1=0.9235
  [ 7/30] ng=1 perm=128 thr=0.6 → mean_F1=0.8668
  [ 8/30] ng=1 perm=128 thr=0.7 → mean_F1=0.8106
  [ 9/30] ng=1 perm=128 thr=0.8 → mean_F1=0.7341
  [10/30] ng=1 perm=128 thr=0.9 → mean_F1=0.5197


KeyboardInterrupt: 

In [ ]:
# ── Summarise tuning: average across all prevalences ─────────────────────────
summary = (
    tuning_df
    .groupby(["ngram_n", "num_perm", "threshold"])[["precision", "recall", "f1"]]
    .mean()
    .reset_index()
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

print("=== Tuning Summary — mean P/R/F1 across all prevalences ===")
display(summary.head(20))

# ── F1 heatmap-style view by threshold × num_perm (best ngram) ───────────────
best_ngram = summary.iloc[0]["ngram_n"]
pivot = (
    summary[summary["ngram_n"] == best_ngram]
    .pivot_table(index="threshold", columns="num_perm", values="f1")
)
print(f"\nF1 heatmap (ngram_n={int(best_ngram)}):")
display(pivot.round(4))

# ── Pick best config ──────────────────────────────────────────────────────────
best = summary.iloc[0]
BEST_NGRAM_N   = int(best["ngram_n"])
BEST_NUM_PERM  = int(best["num_perm"])
BEST_THRESHOLD = float(best["threshold"])

print(f"\n{'='*50}")
print(f"Best config selected:")
print(f"  ngram_n   = {BEST_NGRAM_N}")
print(f"  num_perm  = {BEST_NUM_PERM}")
print(f"  threshold = {BEST_THRESHOLD}")
print(f"  mean_F1   = {best['f1']:.4f}")
print(f"  mean_P    = {best['precision']:.4f}")
print(f"  mean_R    = {best['recall']:.4f}")


In [ ]:
print("=== Final Test Evaluation ===\n")
print(f"Config: ngram_n={BEST_NGRAM_N}, num_perm={BEST_NUM_PERM}, threshold={BEST_THRESHOLD}\n")

test_rows = []
_saved_results = {}    # keep (y_true, y_pred, stream_df) for analysis below

for prev, stream_df in test_streams.items():
    metrics, y_true, y_pred = run_minhash_lsh_stream(
        stream_df, BEST_THRESHOLD, BEST_NUM_PERM, BEST_NGRAM_N
    )
    _saved_results[prev] = (y_true, y_pred, stream_df)

    mod_col = stream_df["modification"].to_numpy()

    # modification-level accuracy (matches author's score() function)
    def _mod_acc(m):
        mask = (mod_col == m)
        if not mask.any():
            return float("nan")
        return float((y_true[mask] == y_pred[mask]).mean())

    acc_mod1 = _mod_acc(1)   # parser duplicates
    acc_mod2 = _mod_acc(2)   # truncation duplicates

    row = {
        "prevalence":             prev,
        "precision":              round(metrics["precision"], 4),
        "recall":                 round(metrics["recall"],    4),
        "f1":                     round(metrics["f1"],        4),
        "tp":                     metrics["tp"],
        "fp":                     metrics["fp"],
        "fn":                     metrics["fn"],
        "n_docs":                 metrics["n_docs"],
        "n_true_dups":            metrics["n_true_duplicates"],
        "acc_parser_dup (mod=1)": round(acc_mod1, 4),
        "acc_trunc_dup  (mod=2)": round(acc_mod2, 4),
        "query_sec":              metrics["query_sec"],
        "insert_sec":             metrics["insert_sec"],
    }
    test_rows.append(row)

    print(f"  prev={prev:.1f} | "
          f"P={metrics['precision']:.4f}  R={metrics['recall']:.4f}  F1={metrics['f1']:.4f} | "
          f"acc_mod1={acc_mod1:.4f}  acc_mod2={acc_mod2:.4f}")

test_results_df = pd.DataFrame(test_rows)
print("\n=== Test Results by Prevalence ===")
display(test_results_df)
print("\n=== Mean across all prevalences ===")
display(test_results_df[["precision", "recall", "f1"]].mean().round(4).to_frame("mean").T)


In [ ]:
print("=== Detailed Breakdown Analysis (test set, prevalence=0.5) ===\n")

prev_focus = 0.5
y_true, y_pred, stream_df = _saved_results[prev_focus]

df_eval = stream_df.copy()
df_eval["y_true"]  = y_true
df_eval["y_pred"]  = y_pred
df_eval["correct"] = (y_true == y_pred).astype(int)

# ── 1. Overall classification report ─────────────────────────────────────────
print("── 1. Overall Classification Report ──")
print(classification_report(
    y_true, y_pred,
    target_names=["Unique (0)", "Duplicate (1)"],
    zero_division=0,
))

# ── 2. Breakdown by modification type ────────────────────────────────────────
print("── 2. By Modification Type ──")
mod_labels = {0: "original  (mod=0)", 1: "parser_dup(mod=1)", 2: "trunc_dup (mod=2)"}
mod_rows = []
for mod_val, mod_name in mod_labels.items():
    sub = df_eval[df_eval["modification"] == mod_val]
    if len(sub) == 0:
        continue
    yt, yp = sub["y_true"].values, sub["y_pred"].values
    r = {
        "modification":     mod_name,
        "n":                len(sub),
        "accuracy":         round((yt == yp).mean(), 4),
        "precision":        round(precision_score(yt, yp, zero_division=0), 4),
        "recall":           round(recall_score(yt, yp, zero_division=0), 4),
        "f1":               round(f1_score(yt, yp, zero_division=0), 4),
    }
    mod_rows.append(r)
display(pd.DataFrame(mod_rows).set_index("modification"))

# ── 3. Recall by truncation percentage (mod=2 only) ──────────────────────────
print("\n── 3. Recall by Truncation Percentage (mod=2) ──")
trunc_sub = df_eval[df_eval["modification"] == 2]
if len(trunc_sub) > 0:
    trunc_recall = (
        trunc_sub.groupby("trunc_percentage")
        .apply(lambda g: recall_score(g["y_true"], g["y_pred"], zero_division=0))
        .reset_index(name="recall")
    )
    trunc_recall.columns = ["trunc_%", "recall"]
    trunc_recall["recall"] = trunc_recall["recall"].round(4)
    display(trunc_recall.set_index("trunc_%"))

# ── 4. Recall by relative truncation location ─────────────────────────────────
print("\n── 4. Recall by Truncation Location (mod=2) ──")
if len(trunc_sub) > 0:
    loc_recall = (
        trunc_sub.groupby("rel_location")
        .apply(lambda g: recall_score(g["y_true"], g["y_pred"], zero_division=0))
        .reset_index(name="recall")
    )
    loc_recall.columns = ["rel_location", "recall"]
    loc_recall["recall"] = loc_recall["recall"].round(4)
    display(loc_recall.set_index("rel_location"))

# ── 5. Parser-duplicate recall by source parser ────────────────────────────────
print("\n── 5. Parser-Duplicate Recall by Parser (mod=1) ──")
parser_sub = df_eval[df_eval["modification"] == 1]
if len(parser_sub) > 0:
    parser_recall = (
        parser_sub.groupby("parser")
        .apply(lambda g: recall_score(g["y_true"], g["y_pred"], zero_division=0))
        .reset_index(name="recall")
    )
    parser_recall.columns = ["parser", "recall"]
    parser_recall["n"] = parser_sub.groupby("parser").size().values
    display(parser_recall.set_index("parser"))

# ── 6. F1 / Precision / Recall across all prevalences (summary table) ─────────
print("\n── 6. Metrics across All Prevalences ──")
display(
    test_results_df[["prevalence", "precision", "recall", "f1",
                     "acc_parser_dup (mod=1)", "acc_trunc_dup  (mod=2)"]]
    .set_index("prevalence")
)


In [ ]:
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

# Tuning results (full grid)
tuning_df.to_csv(out / "tuning_results.csv", index=False)

# Test results (per prevalence)
test_results_df.to_csv(out / "test_results.csv", index=False)

# Best config + summary metrics
best_cfg = {
    "best_ngram_n":        BEST_NGRAM_N,
    "best_num_perm":       BEST_NUM_PERM,
    "best_threshold":      BEST_THRESHOLD,
    "mean_test_f1":        float(test_results_df["f1"].mean()),
    "mean_test_precision": float(test_results_df["precision"].mean()),
    "mean_test_recall":    float(test_results_df["recall"].mean()),
}
with open(out / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_cfg, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {out.resolve()}")
print(f"\nFiles written:")
for p in sorted(out.iterdir()):
    print(f"  {p.name}")

print(f"\n{'='*50}")
print("Best Config Summary:")
for k, v in best_cfg.items():
    print(f"  {k}: {v}")


tuning stream count: 3 | unique docs used: 7918
test stream count  : 3 | unique docs used: 24090
